In [1]:
# Move the root directory to the system path 
import sys
from pathlib import Path

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [2]:
import pandas as pd
from src.vocabulary import Vocabulary
from src.utils import save_vocab

# Load captions from the CSV file
CAPTION_DIR = "../data/raw/flickr30k_images/results.csv"

df = pd.read_csv(CAPTION_DIR, sep = '|')
df.columns = df.columns.str.strip()
df = df.dropna(subset=['comment'])

captions = df.comment.tolist()
captions[:5]

[' Two young guys with shaggy hair look at their hands while hanging out in the yard .',
 ' Two young , White males are outside near many bushes .',
 ' Two men in green shirts are standing in a yard .',
 ' A man in a blue shirt standing in a garden .',
 ' Two friends enjoy time spent together .']

In [3]:
# Verify that the captions are not consist of NaN values
print("NaNs in caption:", df["comment"].isna().sum())
print(df[df["comment"].isna()])   

NaNs in caption: 0
Empty DataFrame
Columns: [image_name, comment_number, comment]
Index: []


In [4]:
# Build the vocabulary
vocab = Vocabulary(freq_threshold=5)

vocab.build_vocabulary(captions)

In [5]:
# Numericalize the first caption
print("Raw caption:", captions[0])
print("Numericalized caption:", vocab.numericalize(captions[0]))

Raw caption:  Two young guys with shaggy hair look at their hands while hanging out in the yard .
Numericalized caption: [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2]


In [6]:
save_vocab(vocab)

## Expand Vocabulary with MS COCO 2017
Merge vocabulary from MS COCO 2017 captions into the existing Flickr30k vocabulary.
Only new tokens appearing with `freq >= freq_threshold` (default 5) in MS COCO that are not in Flickr30k will be appended.

In [ ]:
import json

# Load MS COCO 2017 train captions
COCO_ANNOTATION_PATH = "../data/raw/coco2017/annotations/captions_train2017.json"

with open(COCO_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

coco_captions = [ann["caption"] for ann in coco_data["annotations"]]
print(f"Total COCO train captions loaded: {len(coco_captions)}")
print("Sample COCO captions:", coco_captions[:3])

In [ ]:
# Build union vocabulary: add only tokens from COCO that are not in Flickr30k
flickr_vocab_len = len(vocab)
vocab.build_vocabulary(coco_captions)
coco_union_len = len(vocab)

print(f"Flickr30k vocabulary size: {flickr_vocab_len}")
print(f"New tokens added from MS COCO: {coco_union_len - flickr_vocab_len}")
print(f"Total union vocabulary size: {coco_union_len}")

In [ ]:
# Verify numericalize on a sample COCO caption
print("Sample raw caption:", coco_captions[0])
print("Sample numericalized:", vocab.numericalize(coco_captions[0]))

In [ ]:
# Save the updated union vocabulary
save_vocab(vocab)
print("Updated union vocabulary saved to data/processed/vocab.pickle")